# Transfer Learning: Model3 (ThesisDNN + XGBoost)

**4 experiments, SEED=42, target fine-tune n=1,000 (fair comparison)**

| Variable | Direction | Target |
|---|---|---|
| `cross_A` | DS-1 → DS-2 cross-transfer | Dataset 2 |
| `cross_B` | DS-2 → DS-1 cross-transfer | Dataset 1 |
| `scratch_D1` | No-transfer DS-1 scratch | Dataset 1 |
| `scratch_D2` | No-transfer DS-2 scratch | Dataset 2 |



In [2]:
# ─────────────────────────────────────────────────────────────
# File Paths  (both CSVs must be in the same directory as this notebook)
# ─────────────────────────────────────────────────────────────
SOURCE_ORIG  = "original_preprocess.csv"        # Dataset 1
SOURCE_CLEAN = "clean_ul_with_conditions2.csv"  # Dataset 2

OUTPUT_FIG_DS1 = "transfer_tgt_ds1.png"
OUTPUT_FIG_DS2 = "transfer_tgt_ds2.png"

In [3]:
import os, random, copy, warnings
warnings.filterwarnings("ignore")

import matplotlib
matplotlib.use("Agg")

import numpy  as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.preprocessing  import StandardScaler
from sklearn.model_selection import train_test_split

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

import xgboost as xgb

print("Imports OK")

Imports OK


In [4]:
# ─────────────────────────────────────────────────────────────
# Global Config
# ─────────────────────────────────────────────────────────────
TARGET_COL     = "pm_power"
SEEDS          = [42]             
MIN_SLICE_SIZE = 20

# Hardware-dependent features: excluded in cross-domain transfer
HW_FEATS = ["turbodec_it", "dec_time"]

# DNN hyper-params (source training)
DNN_EPOCHS  = 500     
DNN_PATIENCE= 50      
DNN_LR      = 1e-3
DNN_BATCH   = 32
DNN_WD      = 0.01

# Fine-tune hyper-params
FT_EPOCHS   = 300     
FT_PATIENCE = 40      
FT_LR       = 5e-4
FT_BATCH    = 32

# XGBoost hyper-params
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
XGB_PARAMS = dict(
    objective        = "reg:squarederror",
    eval_metric      = "rmse",
    eta              = 0.15,      # 0.22 → 0.15: slower/more careful boosting
    max_depth        = 5,
    subsample        = 0.85,
    colsample_bytree = 0.9,
    min_child_weight = 3,
    reg_lambda       = 1.0,
    reg_alpha        = 0.1,
    tree_method      = "hist",
    device           = DEVICE,
)
XGB_ROUNDS = 400      # 256 → 400: more trees to compensate smaller eta
XGB_EARLY  = 40       # 30  → 40

print(f"[Config] device={DEVICE} | seeds={SEEDS}")
print(f"[Config] DNN_EPOCHS={DNN_EPOCHS} | DNN_PATIENCE={DNN_PATIENCE}")
print(f"[Config] FT_EPOCHS={FT_EPOCHS}  | FT_PATIENCE={FT_PATIENCE}")
print(f"[Config] XGB_ROUNDS={XGB_ROUNDS} | eta={XGB_PARAMS['eta']}")
print(f"[Config] HW_FEATS excluded: {HW_FEATS}")

[Config] device=cuda | seeds=[42]
[Config] DNN_EPOCHS=500 | DNN_PATIENCE=50
[Config] FT_EPOCHS=300  | FT_PATIENCE=40
[Config] XGB_ROUNDS=400 | eta=0.15
[Config] HW_FEATS excluded: ['turbodec_it', 'dec_time']


In [5]:
# ─────────────────────────────────────────────────────────────
# Reproducibility
# ─────────────────────────────────────────────────────────────
def set_seed(seed=SEEDS[0]):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)
    torch.backends.cudnn.deterministic = True
    torch.backends.cudnn.benchmark     = False

set_seed()

# ─────────────────────────────────────────────────────────────
# Column Mapping  (original_preprocess _ul suffix → clean naming)
# ─────────────────────────────────────────────────────────────
ORIG_TO_CLEAN = {
    "txgain_ul":       "txgain",
    "selected_mcs_ul": "selected_mcs",
    "airtime_ul":      "airtime",
    "nRBs_ul":         "nRBs",
    "mean_snr_ul":     "mean_snr",
    "bler_ul":         "bler",
    "thr_ul":          "thr",
    "bsr_ul":          "bsr",
}

# ─────────────────────────────────────────────────────────────
# Feature Engineering
# ─────────────────────────────────────────────────────────────
def add_feature_engineering(df: pd.DataFrame) -> pd.DataFrame:
    d = df.copy()
    def has(*cols): return all(c in d.columns for c in cols)
    if has("txgain", "airtime"):
        d["txgain_x_airtime"] = d["txgain"] * d["airtime"]
    if has("selected_mcs", "airtime"):
        d["mcs_x_airtime"]    = d["selected_mcs"] * d["airtime"]
    if has("mean_snr", "bler"):
        d["snr_per_bler"]     = d["mean_snr"].astype(float) / (d["bler"].astype(float) + 1e-6)
    if has("thr", "airtime"):
        d["thr_per_airtime"]  = d["thr"].astype(float) / (d["airtime"].astype(float).clip(lower=0.01) + 1e-6)
    return d

In [6]:
# ─────────────────────────────────────────────────────────────
# Feature & Experiment Definitions
# ─────────────────────────────────────────────────────────────
COMMON_BASE = ["txgain", "selected_mcs", "airtime", "nRBs",
               "mean_snr", "bler", "thr", "bsr",
               "turbodec_it", "dec_time", "num_ues"]
COMMON_ENG  = ["txgain_x_airtime", "mcs_x_airtime", "snr_per_bler", "thr_per_airtime"]

EXPERIMENTS = {
    "Transmission Gain": {"slice_col": "txgain",       "base_feats": ["selected_mcs", "airtime", "nRBs"]},
    "MCS":               {"slice_col": "selected_mcs", "base_feats": ["txgain",        "airtime", "nRBs"]},
    "Airtime":           {"slice_col": "airtime",      "base_feats": ["txgain", "selected_mcs", "nRBs"]},
}

def get_feature_cols(exp_cfg, df):
    base  = [c for c in exp_cfg["base_feats"] + COMMON_BASE if c in df.columns]
    eng   = [c for c in COMMON_ENG if c in df.columns]
    feats = list(dict.fromkeys(base + eng))
    return feats

# ─────────────────────────────────────────────────────────────
# Dataset Loaders
# ─────────────────────────────────────────────────────────────
def load_orig(path=SOURCE_ORIG):
    df = pd.read_csv(path)
    df = df.rename(columns=ORIG_TO_CLEAN)
    df = add_feature_engineering(df)
    df = df[df[TARGET_COL] > 0].dropna(subset=[TARGET_COL]).reset_index(drop=True)
    print(f"  [Dataset1/orig]  loaded {len(df):,} rows")
    return df

def load_clean(path=SOURCE_CLEAN):
    df = pd.read_csv(path)
    df = add_feature_engineering(df)
    df = df[df[TARGET_COL] > 0].dropna(subset=[TARGET_COL]).reset_index(drop=True)
    print(f"  [Dataset2/clean] loaded {len(df):,} rows")
    return df

In [7]:
# ─────────────────────────────────────────────────────────────
# Preprocessing Helpers
# ─────────────────────────────────────────────────────────────
def clean_numeric(df, cols):
    d = df.dropna(subset=cols).copy()
    for c in cols:
        d[c] = pd.to_numeric(d[c], errors="coerce")
    return d.dropna(subset=cols).copy()

def winsorize_fit(X, lo=1, hi=99):
    X_w, bounds = X.copy().astype(float), []
    for j in range(X.shape[1]):
        l, h = np.percentile(X_w[:, j], lo), np.percentile(X_w[:, j], hi)
        if h > l: X_w[:, j] = np.clip(X_w[:, j], l, h)
        bounds.append((l, h))
    return X_w, bounds

def winsorize_apply(X, bounds):
    X_w = X.copy().astype(float)
    for j, (l, h) in enumerate(bounds):
        if h > l: X_w[:, j] = np.clip(X_w[:, j], l, h)
    return X_w

def split_train_val_test(df, seed=SEEDS[0], train_r=0.8, val_r=0.1):
    tr, te = train_test_split(df, test_size=1-train_r, random_state=seed)
    tr, va = train_test_split(tr, test_size=val_r,     random_state=seed)
    return tr, va, te

In [8]:
# ─────────────────────────────────────────────────────────────
# ThesisDNN  (input→587→261→186→99→bottleneck(32)→head(1))
# Bottleneck increased from 16 → 32 for richer embeddings
# ─────────────────────────────────────────────────────────────
class TabularDS(Dataset):
    def __init__(self, X, y):
        self.X = torch.tensor(np.asarray(X), dtype=torch.float32)
        self.y = torch.tensor(np.asarray(y), dtype=torch.float32).view(-1, 1)
    def __len__(self):        return len(self.X)
    def __getitem__(self, i): return self.X[i], self.y[i]

class ThesisDNN(nn.Module):
    def __init__(self, input_dim):
        super().__init__()
        self.fc1        = nn.Linear(input_dim, 587)
        self.fc2        = nn.Linear(587, 261)
        self.fc3        = nn.Linear(261, 186)
        self.fc4        = nn.Linear(186,  99)
        self.bottleneck = nn.Linear( 99,  32)  # 16 → 32
        self.head       = nn.Linear( 32,   1)

    def forward(self, x):
        h   = F.relu(self.fc1(x))
        h   = F.relu(self.fc2(h))
        h   = F.relu(self.fc3(h))
        h   = F.relu(self.fc4(h))
        emb = self.bottleneck(h)
        out = self.head(F.relu(emb))
        return out, emb

In [9]:
# ─────────────────────────────────────────────────────────────
# DNN Training
# ─────────────────────────────────────────────────────────────
def train_dnn_generic(X_tr, y_tr, X_va, y_va,
                      input_dim, model=None,
                      epochs=DNN_EPOCHS, batch=DNN_BATCH,
                      lr=DNN_LR, wd=DNN_WD, patience=DNN_PATIENCE,
                      verbose_every=100, seed=SEEDS[0]):
    set_seed(seed)
    if model is None:
        model = ThesisDNN(input_dim).to(DEVICE)
    else:
        model = model.to(DEVICE)

    loader  = DataLoader(TabularDS(X_tr, y_tr), batch_size=batch, shuffle=True)
    loss_fn = nn.MSELoss()
    # Only optimize parameters with requires_grad=True (handles frozen layers)
    trainable_params = [p for p in model.parameters() if p.requires_grad]
    optimizer = torch.optim.Adam(trainable_params, lr=lr, weight_decay=wd)

    X_va_t = torch.tensor(X_va, dtype=torch.float32).to(DEVICE)
    y_va_t = torch.tensor(y_va, dtype=torch.float32).view(-1, 1).to(DEVICE)

    best_val, best_state, no_impr = float("inf"), None, 0
    for ep in range(1, epochs + 1):
        model.train()
        for xb, yb in loader:
            xb, yb = xb.to(DEVICE), yb.to(DEVICE)
            optimizer.zero_grad()
            pred, _ = model(xb)
            loss_fn(pred, yb).backward()
            optimizer.step()

        model.eval()
        with torch.no_grad():
            vp, _ = model(X_va_t)
            vloss = loss_fn(vp, y_va_t).item()

        if vloss < best_val - 1e-8:
            best_val, best_state, no_impr = vloss, {k: v.cpu().clone()
                for k, v in model.state_dict().items()}, 0
        else:
            no_impr += 1

        if verbose_every and ep % verbose_every == 0:
            print(f"    ep {ep:04d}  val_MSE={vloss:.5f}  no_impr={no_impr}")

        if no_impr >= patience:
            break

    model.load_state_dict(best_state)
    model.eval()
    return model


@torch.no_grad()
def extract_embeddings(model, X_s, batch_size=1024):
    model.eval()
    X_t  = torch.tensor(X_s, dtype=torch.float32)
    embs = []
    for i in range(0, len(X_t), batch_size):
        _, emb = model(X_t[i:i+batch_size].to(DEVICE))
        embs.append(emb.cpu().numpy())
    return np.vstack(embs)

In [10]:
# ─────────────────────────────────────────────────────────────
# XGBoost Training
# ─────────────────────────────────────────────────────────────
def train_xgb(emb_tr, y_tr, emb_va, y_va,
              xgb_model=None, rounds=XGB_ROUNDS, seed=SEEDS[0]):
    params  = {**XGB_PARAMS, "seed": seed}
    dtrain  = xgb.DMatrix(emb_tr, label=y_tr)
    dval    = xgb.DMatrix(emb_va, label=y_va)
    booster = xgb.train(
        params, dtrain,
        num_boost_round       = rounds,
        evals                 = [(dval, "val")],
        early_stopping_rounds = XGB_EARLY,
        verbose_eval          = False,
        xgb_model             = xgb_model,
    )
    return booster

In [11]:
# ─────────────────────────────────────────────────────────────
# Metrics
# ─────────────────────────────────────────────────────────────
def relative_errors(y_true, y_pred, eps=1e-3):
    """Point-wise relative error (%)."""
    yt = np.asarray(y_true).ravel()
    yp = np.asarray(y_pred).ravel()
    return np.abs(yt - yp) / (np.abs(yt) + eps) * 100.0


def slice_re_stats(df_eval, y_pred, slice_col):
    """
    Per-slice (mean_re, std_re, max_re, min_re).
    Returns DataFrame: slice_val | mean_re | std_re | max_re | min_re
    """
    y_true = df_eval[TARGET_COL].values
    re_arr = relative_errors(y_true, y_pred)
    df_tmp = df_eval[[slice_col]].copy().reset_index(drop=True)
    df_tmp["re"] = re_arr
    agg = (df_tmp.groupby(slice_col)["re"]
                 .agg(mean_re="mean", std_re="std", max_re="max", min_re="min")
                 .reset_index()
                 .rename(columns={slice_col: "slice_val"}))
    agg["std_re"] = agg["std_re"].fillna(0.0)
    return agg


def slice_power_stats(df_eval, y_pred, slice_col):
    """
    Per-slice mean actual and predicted power.
    Returns DataFrame: slice_val | mean_actual | mean_pred
    """
    df_tmp = df_eval[[slice_col, TARGET_COL]].copy().reset_index(drop=True)
    df_tmp["pred"] = np.asarray(y_pred).ravel()
    agg = (df_tmp.groupby(slice_col)
                 .agg(mean_actual=(TARGET_COL, "mean"),
                      mean_pred=("pred", "mean"))
                 .reset_index()
                 .rename(columns={slice_col: "slice_val"}))
    return agg


In [12]:
# ─────────────────────────────────────────────────────────────
# run_transfer()  —  SOURCE → TARGET cross-transfer
# Returns: (re_stats_df, power_stats_df, re_arr)
# ─────────────────────────────────────────────────────────────
def run_transfer(df_src, df_tgt, exp_name, exp_cfg, seed=SEEDS[0]):
    print(f"\n  ── {exp_name} [cross-transfer | seed={seed}] ──")
    slice_col = exp_cfg["slice_col"]

    feat_src = get_feature_cols(exp_cfg, df_src)
    feat_tgt = get_feature_cols(exp_cfg, df_tgt)
    feats = [f for f in feat_src if f in feat_tgt and f not in HW_FEATS]
    print(f"    feats ({len(feats)}): {feats}")

    d_src = clean_numeric(df_src, [c for c in feats + [slice_col, TARGET_COL] if c in df_src.columns])
    d_tgt = clean_numeric(df_tgt, [c for c in feats + [slice_col, TARGET_COL] if c in df_tgt.columns])

    s_tr, s_va, _ = split_train_val_test(d_src, seed=seed)
    X_s_tr = s_tr[feats].values.astype(float)
    y_s_tr = s_tr[TARGET_COL].values.astype(float)
    X_s_va = s_va[feats].values.astype(float)
    y_s_va = s_va[TARGET_COL].values.astype(float)

    X_s_tr_w, bounds_s = winsorize_fit(X_s_tr)
    X_s_va_w           = winsorize_apply(X_s_va, bounds_s)
    sc_src             = StandardScaler()
    X_s_tr_s           = sc_src.fit_transform(X_s_tr_w)
    X_s_va_s           = sc_src.transform(X_s_va_w)

    print(f"    [Source DNN] training on {len(s_tr)+len(s_va)} samples …")
    dnn_src = train_dnn_generic(
        X_s_tr_s, y_s_tr, X_s_va_s, y_s_va,
        input_dim=len(feats), epochs=DNN_EPOCHS, batch=DNN_BATCH,
        lr=DNN_LR, wd=DNN_WD, patience=DNN_PATIENCE, seed=seed)

    t_tr, t_va, t_te = split_train_val_test(d_tgt, seed=seed)
    print(f"    [Target] fine-tune train={len(t_tr)}, val={len(t_va)}, test={len(t_te)}")

    X_t_tr = t_tr[feats].values.astype(float)
    y_t_tr = t_tr[TARGET_COL].values.astype(float)
    X_t_va = t_va[feats].values.astype(float)
    y_t_va = t_va[TARGET_COL].values.astype(float)
    X_t_te = t_te[feats].values.astype(float)

    X_t_tr_w, bounds_t = winsorize_fit(X_t_tr)
    X_t_va_w           = winsorize_apply(X_t_va, bounds_t)
    X_t_te_w           = winsorize_apply(X_t_te, bounds_t)
    sc_tgt             = StandardScaler()
    X_t_tr_s           = sc_tgt.fit_transform(X_t_tr_w)
    X_t_va_s           = sc_tgt.transform(X_t_va_w)
    X_t_te_s           = sc_tgt.transform(X_t_te_w)

    dnn_ft = copy.deepcopy(dnn_src)
    for name, param in dnn_ft.named_parameters():
        if name.startswith(("fc1.", "fc2.")):
            param.requires_grad = False
    n_trainable = sum(p.numel() for p in dnn_ft.parameters() if p.requires_grad)
    print(f"    [Fine-tune DNN] {len(t_tr)} samples, trainable params: {n_trainable} …")
    dnn_ft = train_dnn_generic(
        X_t_tr_s, y_t_tr, X_t_va_s, y_t_va,
        input_dim=len(feats), model=dnn_ft,
        epochs=FT_EPOCHS, batch=FT_BATCH,
        lr=FT_LR, wd=DNN_WD, patience=FT_PATIENCE, seed=seed)
    for param in dnn_ft.parameters():
        param.requires_grad = True

    emb_t_tr = extract_embeddings(dnn_ft, X_t_tr_s)
    emb_t_va = extract_embeddings(dnn_ft, X_t_va_s)
    emb_t_te = extract_embeddings(dnn_ft, X_t_te_s)

    print(f"    [Target XGB] training …")
    xgb_ft = train_xgb(emb_t_tr, y_t_tr, emb_t_va, y_t_va,
                       xgb_model=None, rounds=XGB_ROUNDS, seed=seed)

    y_pred = xgb_ft.predict(xgb.DMatrix(emb_t_te))
    y_pred = np.clip(y_pred, y_t_tr.min() * 0.9, y_t_tr.max() * 1.1)

    df_eval = t_te.copy().reset_index(drop=True)
    re_arr  = relative_errors(df_eval[TARGET_COL].values, y_pred)
    stats   = slice_re_stats(df_eval, y_pred, slice_col)
    power   = slice_power_stats(df_eval, y_pred, slice_col)

    print(f"    Cross-transfer MRE = {re_arr.mean():.2f}%  "
          f"(max={re_arr.max():.1f}%  min={re_arr.min():.1f}%)")
    return stats, power, re_arr


In [13]:
# ─────────────────────────────────────────────────────────────
# run_scratch()  —  No-Transfer Scratch Baseline
# Returns: (re_stats_df, power_stats_df, re_arr)
# ─────────────────────────────────────────────────────────────
def run_scratch(df_tgt, exp_name, exp_cfg, seed=SEEDS[0]):
    print(f"\n  ── {exp_name} [no-transfer scratch | seed={seed}] ──")
    slice_col = exp_cfg["slice_col"]

    feats_all = get_feature_cols(exp_cfg, df_tgt)
    feats = [f for f in feats_all if f not in HW_FEATS]
    print(f"    feats ({len(feats)}): {feats}")

    d_tgt = clean_numeric(df_tgt, [c for c in feats + [slice_col, TARGET_COL] if c in df_tgt.columns])

    t_tr, t_va, t_te = split_train_val_test(d_tgt, seed=seed)
    print(f"    [Target] scratch train={len(t_tr)}, val={len(t_va)}, test={len(t_te)}")

    X_t_tr = t_tr[feats].values.astype(float)
    y_t_tr = t_tr[TARGET_COL].values.astype(float)
    X_t_va = t_va[feats].values.astype(float)
    y_t_va = t_va[TARGET_COL].values.astype(float)
    X_t_te = t_te[feats].values.astype(float)

    X_t_tr_w, bounds_t = winsorize_fit(X_t_tr)
    X_t_va_w           = winsorize_apply(X_t_va, bounds_t)
    X_t_te_w           = winsorize_apply(X_t_te, bounds_t)
    sc_tgt             = StandardScaler()
    X_t_tr_s           = sc_tgt.fit_transform(X_t_tr_w)
    X_t_va_s           = sc_tgt.transform(X_t_va_w)
    X_t_te_s           = sc_tgt.transform(X_t_te_w)

    print(f"    [Scratch DNN] training from random init on {len(t_tr)} samples …")
    dnn_scratch = train_dnn_generic(
        X_t_tr_s, y_t_tr, X_t_va_s, y_t_va,
        input_dim=len(feats),
        epochs=DNN_EPOCHS, batch=DNN_BATCH,
        lr=DNN_LR, wd=DNN_WD, patience=DNN_PATIENCE,
        seed=seed)

    emb_tr = extract_embeddings(dnn_scratch, X_t_tr_s)
    emb_va = extract_embeddings(dnn_scratch, X_t_va_s)
    emb_te = extract_embeddings(dnn_scratch, X_t_te_s)

    print(f"    [Scratch XGB] training …")
    xgb_scratch = train_xgb(emb_tr, y_t_tr, emb_va, y_t_va,
                            xgb_model=None, rounds=XGB_ROUNDS, seed=seed)

    y_pred = xgb_scratch.predict(xgb.DMatrix(emb_te))
    y_pred = np.clip(y_pred, y_t_tr.min() * 0.9, y_t_tr.max() * 1.1)

    df_eval = t_te.copy().reset_index(drop=True)
    re_arr  = relative_errors(df_eval[TARGET_COL].values, y_pred)
    stats   = slice_re_stats(df_eval, y_pred, slice_col)
    power   = slice_power_stats(df_eval, y_pred, slice_col)

    print(f"    Scratch MRE = {re_arr.mean():.2f}%  "
          f"(max={re_arr.max():.1f}%  min={re_arr.min():.1f}%)")
    return stats, power, re_arr


In [14]:
# ─────────────────────────────────────────────────────────────
# Load Datasets
# ─────────────────────────────────────────────────────────────
print("=" * 60)
print("Loading datasets …")
print("=" * 60)
df_orig  = load_orig()
df_clean = load_clean()

Loading datasets …
  [Dataset1/orig]  loaded 17,501 rows
  [Dataset2/clean] loaded 5,769 rows


In [15]:
# ─────────────────────────────────────────────────────────────
# Run all experiments
#   all_results[seed][key][exp_name] = re_stats_df
#   all_power  [seed][key][exp_name] = power_stats_df
#   all_re     [seed][key][exp_name] = re_arr (per-sample)
# ─────────────────────────────────────────────────────────────
all_results = {}
all_power   = {}
all_re      = {}

for seed in SEEDS:

    all_results[seed] = {"cross_A": {}, "cross_B": {}, "scratch_D1": {}, "scratch_D2": {}}
    all_power[seed]   = {"cross_A": {}, "cross_B": {}, "scratch_D1": {}, "scratch_D2": {}}
    all_re[seed]      = {"cross_A": {}, "cross_B": {}, "scratch_D1": {}, "scratch_D2": {}}

    for exp_name, exp_cfg in EXPERIMENTS.items():
        print(f"\n{'='*55}\n  Experiment: {exp_name}\n{'='*55}")

        print("\n  [1] cross_A — DS-1 → DS-2")
        s, p, r = run_transfer(df_orig, df_clean, exp_name, exp_cfg, seed=seed)
        all_results[seed]["cross_A"][exp_name] = s
        all_power[seed]["cross_A"][exp_name]   = p
        all_re[seed]["cross_A"][exp_name]      = r

        print("\n  [2] cross_B — DS-2 → DS-1")
        s, p, r = run_transfer(df_clean, df_orig, exp_name, exp_cfg, seed=seed)
        all_results[seed]["cross_B"][exp_name] = s
        all_power[seed]["cross_B"][exp_name]   = p
        all_re[seed]["cross_B"][exp_name]      = r

        print("\n  [3] scratch_D1 — DS-1 no-transfer")
        s, p, r = run_scratch(df_orig, exp_name, exp_cfg, seed=seed)
        all_results[seed]["scratch_D1"][exp_name] = s
        all_power[seed]["scratch_D1"][exp_name]   = p
        all_re[seed]["scratch_D1"][exp_name]      = r

        print("\n  [4] scratch_D2 — DS-2 no-transfer")
        s, p, r = run_scratch(df_clean, exp_name, exp_cfg, seed=seed)
        all_results[seed]["scratch_D2"][exp_name] = s
        all_power[seed]["scratch_D2"][exp_name]   = p
        all_re[seed]["scratch_D2"][exp_name]      = r

print(f"\n[Done] All {len(SEEDS)} seeds × {len(EXPERIMENTS)} experiments finished.")



  Experiment: Transmission Gain

  [1] cross_A — DS-1 → DS-2

  ── Transmission Gain [cross-transfer | seed=42] ──
    feats (13): ['selected_mcs', 'airtime', 'nRBs', 'txgain', 'mean_snr', 'bler', 'thr', 'bsr', 'num_ues', 'txgain_x_airtime', 'mcs_x_airtime', 'snr_per_bler', 'thr_per_airtime']
    [Source DNN] training on 14000 samples …
    ep 0100  val_MSE=0.10574  no_impr=12
    [Target] fine-tune train=4153, val=462, test=1154
    [Fine-tune DNN] 4153 samples, trainable params: 70478 …
    [Target XGB] training …
    Cross-transfer MRE = 2.13%  (max=19.2%  min=0.0%)

  [2] cross_B — DS-2 → DS-1

  ── Transmission Gain [cross-transfer | seed=42] ──
    feats (13): ['selected_mcs', 'airtime', 'nRBs', 'txgain', 'mean_snr', 'bler', 'thr', 'bsr', 'num_ues', 'txgain_x_airtime', 'mcs_x_airtime', 'snr_per_bler', 'thr_per_airtime']
    [Source DNN] training on 4615 samples …
    ep 0100  val_MSE=0.08224  no_impr=41
    [Target] fine-tune train=12600, val=1400, test=3501
    [Fine-tune DNN] 

In [17]:
# ─────────────────────────────────────────────────────────────
# Aggregate results across seeds
# ─────────────────────────────────────────────────────────────
def agg_seeds(key, exp_name):
    dfs = [all_results[seed][key][exp_name].copy() for seed in SEEDS]
    combined = pd.concat(dfs, ignore_index=True)
    return (combined.groupby("slice_val")
            .agg(mean_re_mean=("mean_re", "mean"),
                 std_re_mean =("std_re",  "mean"),
                 max_re_mean =("max_re",  "mean"),
                 min_re_mean =("min_re",  "mean"))
            .reset_index())

def agg_power_seeds(key, exp_name):
    dfs = [all_power[seed][key][exp_name].copy() for seed in SEEDS]
    combined = pd.concat(dfs, ignore_index=True)
    return (combined.groupby("slice_val")
            .agg(mean_actual=("mean_actual", "mean"),
                 mean_pred  =("mean_pred",   "mean"))
            .reset_index())

cross_A    = {exp: agg_seeds("cross_A",    exp) for exp in EXPERIMENTS}
cross_B    = {exp: agg_seeds("cross_B",    exp) for exp in EXPERIMENTS}
scratch_D1 = {exp: agg_seeds("scratch_D1", exp) for exp in EXPERIMENTS}
scratch_D2 = {exp: agg_seeds("scratch_D2", exp) for exp in EXPERIMENTS}

power_cross_A    = {exp: agg_power_seeds("cross_A",    exp) for exp in EXPERIMENTS}
power_cross_B    = {exp: agg_power_seeds("cross_B",    exp) for exp in EXPERIMENTS}
power_scratch_D1 = {exp: agg_power_seeds("scratch_D1", exp) for exp in EXPERIMENTS}
power_scratch_D2 = {exp: agg_power_seeds("scratch_D2", exp) for exp in EXPERIMENTS}

print("Aggregation done.")
for exp in EXPERIMENTS:
    cb = cross_B[exp];  sd1 = scratch_D1[exp]
    ca = cross_A[exp];  sd2 = scratch_D2[exp]
    print(f"\n{exp}")
    print(f"  DS2→DS1 cross  MRE = {cb['mean_re_mean'].mean():.3f}%")
    print(f"  DS1 scratch    MRE = {sd1['mean_re_mean'].mean():.3f}%")
    print(f"  DS1→DS2 cross  MRE = {ca['mean_re_mean'].mean():.3f}%")
    print(f"  DS2 scratch    MRE = {sd2['mean_re_mean'].mean():.3f}%")


Aggregation done.

Transmission Gain
  DS2→DS1 cross  MRE = 2.012%
  DS1 scratch    MRE = 1.806%
  DS1→DS2 cross  MRE = 2.159%
  DS2 scratch    MRE = 1.907%

MCS
  DS2→DS1 cross  MRE = 1.869%
  DS1 scratch    MRE = 1.808%
  DS1→DS2 cross  MRE = 2.296%
  DS2 scratch    MRE = 1.869%

Airtime
  DS2→DS1 cross  MRE = 1.995%
  DS1 scratch    MRE = 1.817%
  DS1→DS2 cross  MRE = 2.172%
  DS2 scratch    MRE = 1.916%


In [18]:
# ─────────────────────────────────────────────────────────────
# Figure 1 — Target = Dataset 1 (original_preprocess)
# Lines = mean across seeds
# ─────────────────────────────────────────────────────────────
EXP_NAMES  = list(EXPERIMENTS.keys())
X_LABELS   = {
    "Transmission Gain": "Transmission gain (dB)",
    "MCS":               "MCS Index",
    "Airtime":           "Airtime Ratio",
}
ROW_LABELS  = ["Relative Error",   "Mean Relative Error", "Max Relative Error", "Min Relative Error"]
METRIC_KEYS = ["mean_re_mean",     "mean_re_mean",        "max_re_mean",        "min_re_mean"]

COLOR_BLUE = "#1f77b4"
COLOR_RED  = "#d62728"

fig1, axes1 = plt.subplots(4, 3, figsize=(15, 16), constrained_layout=True)
fig1.suptitle(
    "Transfer Learning — Target: Dataset 1 (original_preprocess) | Model3 (DNN + XGBoost)\n"
    f"Mean across seeds {SEEDS}",
    fontsize=12, fontweight="bold"
)

for row_idx in range(4):
    mkey  = METRIC_KEYS[row_idx]
    ylabel = f"{ROW_LABELS[row_idx]} (%)"

    for col_idx, exp_name in enumerate(EXP_NAMES):
        ax = axes1[row_idx, col_idx]

        st_blue = cross_B[exp_name].sort_values("slice_val")
        st_red  = scratch_D1[exp_name].sort_values("slice_val")

        x_b, y_b = st_blue["slice_val"].values, st_blue[mkey].values
        x_r, y_r = st_red["slice_val"].values,  st_red[mkey].values

        lbl_blue = f"Cross-transfer: DS-2\u2192DS-1 (full data)"
        lbl_red  = f"No-transfer: DS-1 scratch (full data)"

        ax.plot(x_b, y_b, color=COLOR_BLUE, lw=2.2, marker="o", ms=4, label=lbl_blue)
        ax.plot(x_r, y_r, color=COLOR_RED,  lw=2.2, marker="s", ms=4, label=lbl_red)

        ax.set_xlabel(X_LABELS[exp_name], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.tick_params(labelsize=8)

        if row_idx == 0:
            ax.set_title(exp_name, fontsize=10, fontweight="bold", pad=6)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8.5, loc="upper right", framealpha=0.9, edgecolor="gray")

plt.savefig(OUTPUT_FIG_DS1, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_FIG_DS1}")
plt.show()

Saved → transfer_tgt_ds1.png


In [19]:
# ─────────────────────────────────────────────────────────────
# Figure 2 — Target = Dataset 2 (clean_ul_with_conditions2)
# Lines = mean across seeds
# ─────────────────────────────────────────────────────────────
fig2, axes2 = plt.subplots(4, 3, figsize=(15, 16), constrained_layout=True)
fig2.suptitle(
    "Transfer Learning — Target: Dataset 2 (clean_ul_with_conditions2) | Model3 (DNN + XGBoost)\n"
    f"Mean across seeds {SEEDS}",
    fontsize=12, fontweight="bold"
)

for row_idx in range(4):
    mkey  = METRIC_KEYS[row_idx]
    ylabel = f"{ROW_LABELS[row_idx]} (%)"

    for col_idx, exp_name in enumerate(EXP_NAMES):
        ax = axes2[row_idx, col_idx]

        st_blue = cross_A[exp_name].sort_values("slice_val")
        st_red  = scratch_D2[exp_name].sort_values("slice_val")

        x_b, y_b = st_blue["slice_val"].values, st_blue[mkey].values
        x_r, y_r = st_red["slice_val"].values,  st_red[mkey].values

        lbl_blue = f"Cross-transfer: DS-1\u2192DS-2 (full data)"
        lbl_red  = f"No-transfer: DS-2 scratch (full data)"

        ax.plot(x_b, y_b, color=COLOR_BLUE, lw=2.2, marker="o", ms=4, label=lbl_blue)
        ax.plot(x_r, y_r, color=COLOR_RED,  lw=2.2, marker="s", ms=4, label=lbl_red)

        ax.set_xlabel(X_LABELS[exp_name], fontsize=9)
        ax.set_ylabel(ylabel, fontsize=9)
        ax.set_ylim(bottom=0)
        ax.grid(True, alpha=0.25, linestyle="--")
        ax.tick_params(labelsize=8)

        if row_idx == 0:
            ax.set_title(exp_name, fontsize=10, fontweight="bold", pad=6)
        if row_idx == 0 and col_idx == 0:
            ax.legend(fontsize=8.5, loc="upper right", framealpha=0.9, edgecolor="gray")

plt.savefig(OUTPUT_FIG_DS2, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_FIG_DS2}")
plt.show()

Saved → transfer_tgt_ds2.png


In [20]:
# ─────────────────────────────────────────────────────────────
# Figure 3 — Metric Heatmap Summary  (2×2 per subplot)
# Values = mean across seeds
# ─────────────────────────────────────────────────────────────
OUTPUT_HEATMAP = "transfer_metric_heatmap.png"

HEATMAP_METRICS = [
    ("mean_re_mean", "mean", "Mean Relative Error (%)"),
    ("max_re_mean",  "max",  "Max Relative Error (%)"),
    ("min_re_mean",  "min",  "Min Relative Error (%)"),
]

COL_CMAPS = ["Blues", "YlOrBr", "Greens"]

def scalar_metric(stats_df, col, agg):
    if agg == "mean": return float(stats_df[col].mean())
    elif agg == "max": return float(stats_df[col].max())
    elif agg == "min": return float(stats_df[col].min())

fig3, axes3 = plt.subplots(3, 3, figsize=(13, 11), constrained_layout=True)
fig3.suptitle(
    f"Transfer Learning Metric Summary — Model3 (DNN + XGBoost)\n"
    f"Mean across seeds {SEEDS}",
    fontsize=12, fontweight="bold"
)

for row_idx, (col_key, agg_fn, metric_label) in enumerate(HEATMAP_METRICS):
    for col_idx, exp_name in enumerate(EXP_NAMES):
        ax   = axes3[row_idx, col_idx]
        cmap = COL_CMAPS[col_idx]

        mat = np.array([
            [scalar_metric(scratch_D1[exp_name], col_key, agg_fn),
             scalar_metric(cross_B[exp_name],    col_key, agg_fn)],
            [scalar_metric(cross_A[exp_name],    col_key, agg_fn),
             scalar_metric(scratch_D2[exp_name], col_key, agg_fn)],
        ])

        im = ax.imshow(mat, cmap=cmap, aspect="auto",
                       vmin=mat.min() * 0.95, vmax=mat.max() * 1.05)

        for ti in range(2):
            for si in range(2):
                val = mat[ti, si]
                text_color = "white" if val > (mat.min() + 0.6 * (mat.max() - mat.min())) else "black"
                ax.text(si, ti, f"{val:.3f}", ha="center", va="center",
                        fontsize=11, fontweight="bold", color=text_color)

        ax.set_xticks([0, 1]);  ax.set_xticklabels(["D1", "D2"], fontsize=9)
        ax.set_yticks([0, 1]);  ax.set_yticklabels(["D1", "D2"], fontsize=9)
        ax.set_xlabel("Source", fontsize=9)
        ax.set_ylabel("Target", fontsize=9)
        ax.set_title(f"{metric_label.split(' (')[0]} — {exp_name}", fontsize=9, pad=5)

        cbar = fig3.colorbar(im, ax=ax, fraction=0.046, pad=0.04)
        cbar.ax.tick_params(labelsize=7)

plt.savefig(OUTPUT_HEATMAP, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_HEATMAP}")
plt.show()

Saved → transfer_metric_heatmap.png


In [39]:
# ─────────────────────────────────────────────────────────────
# Figure 4 — Bar Chart: Mean Relative Error per Feature Value
#
# Layout: 2 rows × 3 columns
#   Row 0: Target = Dataset 1
#   Row 1: Target = Dataset 2
#   Col 0: Txgain  |  Col 1: MCS  |  Col 2: Airtime
#
# bar height = mean relative error (%)
# black error bar = ± std (standard deviation)
# ─────────────────────────────────────────────────────────────
OUTPUT_FIG_BAR = "transfer_bar_mre.png"

EXP_LIST    = list(EXPERIMENTS.keys())   # ["Transmission Gain", "MCS", "Airtime"]
COL_XLABELS = ["Transmission gain (dB)", "MCS Index", "Airtime Ratio"]
COL_TITLES  = ["Transmission gain", "MCS", "Airtime"]
ROW_TITLES  = ["Target = Dataset 1", "Target = Dataset 2"]

COLOR_DS1 = "#1f77b4"
COLOR_DS2 = "#ff7f0e"

# Txgain filters per row
# Row 0 (Target DS1): DS1 has 27 discrete values — pick these 10
TXGAIN_FILTER_DS1 = {23, 28, 33, 38, 43, 48, 53, 58, 63, 68}
# Row 1 (Target DS2): DS2 has 55 continuous values (20-74) — pick 10 evenly spaced
TXGAIN_FILTER_DS2 = set(np.round(np.linspace(20, 74, 10)).astype(int))

TXGAIN_FILTERS = [TXGAIN_FILTER_DS1, TXGAIN_FILTER_DS2]

# row=0 (Target DS1): Source DS1=scratch_D1, Source DS2=cross_B (DS2→DS1)
# row=1 (Target DS2): Source DS1=cross_A (DS1→DS2), Source DS2=scratch_D2
DATA_MAP = [
    (scratch_D1, cross_B),
    (cross_A,    scratch_D2),
]
SRC_LABELS = [
    ("Source DS1 (no transfer)", "Source DS2 → DS1"),
    ("Source DS1 → DS2",        "Source DS2 (no transfer)"),
]

fig4, axes4 = plt.subplots(2, 3, figsize=(15, 8), constrained_layout=True)
fig4.suptitle(
    "Transfer Learning — Mean Relative Error by Feature Value\n"
    "Model3 (DNN + XGBoost)",
    fontsize=12, fontweight="bold"
)

for row_idx in range(2):
    dict_ds1, dict_ds2 = DATA_MAP[row_idx]
    lbl_ds1,  lbl_ds2  = SRC_LABELS[row_idx]

    for col_idx, exp_name in enumerate(EXP_LIST):
        ax = axes4[row_idx, col_idx]

        df1 = dict_ds1[exp_name].sort_values("slice_val")
        df2 = dict_ds2[exp_name].sort_values("slice_val")

        common_vals = sorted(set(df1["slice_val"]) & set(df2["slice_val"]))

        # Filter txgain to 10 selected values for both rows
        if exp_name == "Transmission Gain":
            common_vals = [v for v in common_vals if v in TXGAIN_FILTERS[row_idx]]

        df1 = df1[df1["slice_val"].isin(common_vals)].set_index("slice_val").loc[common_vals]
        df2 = df2[df2["slice_val"].isin(common_vals)].set_index("slice_val").loc[common_vals]

        x = np.arange(len(common_vals))
        w = 0.38

        y1 = df1["mean_re_mean"].values
        s1 = df1["std_re_mean"].values

        y2 = df2["mean_re_mean"].values
        s2 = df2["std_re_mean"].values

        err_kw = dict(color="black", lw=1.2, capsize=3, capthick=1.2)

        ax.bar(x - w / 2, y1, w,
               color=COLOR_DS1, alpha=0.85, label=lbl_ds1,
               yerr=s1, error_kw=err_kw)
        ax.bar(x + w / 2, y2, w,
               color=COLOR_DS2, alpha=0.85, label=lbl_ds2,
               yerr=s2, error_kw=err_kw)

        tick_labels = [f"{v:.3g}" if isinstance(v, float) else str(int(v))
                       for v in common_vals]
        ax.set_xticks(x)
        ax.set_xticklabels(tick_labels, fontsize=7, rotation=45, ha="right")
        ax.set_xlabel(COL_XLABELS[col_idx], fontsize=9)
        ax.set_ylim(bottom=0)
        ax.grid(True, axis="y", alpha=0.25, linestyle="--")
        ax.tick_params(labelsize=8)

        if col_idx == 0:
            ax.set_ylabel(f"{ROW_TITLES[row_idx]}\nMean Relative Error (%)", fontsize=9)
        else:
            ax.set_ylabel("Mean Relative Error (%)", fontsize=9)

        if row_idx == 0:
            ax.set_title(COL_TITLES[col_idx], fontsize=10, fontweight="bold", pad=6)

        ax.legend(fontsize=8, loc="upper right", framealpha=0.9, edgecolor="gray")

plt.savefig(OUTPUT_FIG_BAR, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_FIG_BAR}")
plt.show()

Saved → transfer_bar_mre.png


In [31]:
# ─────────────────────────────────────────────────────────────
# Figure 5 — Transmission Gain 
#  Target = Dataset 1  |  
#  Target = Dataset 2
# ─────────────────────────────────────────────────────────────
OUTPUT_FIG_TXGAIN = "transfer_bar_txgain.png"

exp_name = "Transmission Gain"

fig5, axes5 = plt.subplots(2, 1, figsize=(22, 12), constrained_layout=True)
fig5.suptitle(
    "Transfer Learning — Transmission Gain (txgain)\n"
    "Mean Relative Error ± Std  |  Model3 (DNN + XGBoost)",
    fontsize=14, fontweight="bold"
)

panel_cfg = [
    (axes5[0],
     scratch_D1, cross_B,
     "Source DS1 (no transfer)", "Source DS2 → DS1",
     "Target = Dataset 1"),
    (axes5[1],
     cross_A, scratch_D2,
     "Source DS1 → DS2", "Source DS2 (no transfer)",
     "Target = Dataset 2"),
]

for ax, dict_ds1, dict_ds2, lbl_ds1, lbl_ds2, panel_title in panel_cfg:
    df1 = dict_ds1[exp_name].sort_values("slice_val")
    df2 = dict_ds2[exp_name].sort_values("slice_val")

    common_vals = sorted(set(df1["slice_val"]) & set(df2["slice_val"]))
    df1 = df1[df1["slice_val"].isin(common_vals)].set_index("slice_val").loc[common_vals]
    df2 = df2[df2["slice_val"].isin(common_vals)].set_index("slice_val").loc[common_vals]

    x = np.arange(len(common_vals))
    w = 0.38

    y1 = df1["mean_re_mean"].values;  s1 = df1["std_re_mean"].values
    y2 = df2["mean_re_mean"].values;  s2 = df2["std_re_mean"].values

    err_kw = dict(color="black", lw=1.4, capsize=4, capthick=1.4)

    ax.bar(x - w / 2, y1, w, color="#1f77b4", alpha=0.85, label=lbl_ds1,
           yerr=s1, error_kw=err_kw)
    ax.bar(x + w / 2, y2, w, color="#ff7f0e", alpha=0.85, label=lbl_ds2,
           yerr=s2, error_kw=err_kw)

    tick_labels = [f"{v:.3g}" if isinstance(v, float) else str(int(v))
                   for v in common_vals]
    ax.set_xticks(x)
    ax.set_xticklabels(tick_labels, fontsize=9, rotation=90, ha="center")
    ax.set_xlabel("txgain (dBm)", fontsize=12)
    ax.set_ylabel("Mean Relative Error (%)", fontsize=12)
    ax.set_title(panel_title, fontsize=13, fontweight="bold", pad=8)
    ax.set_xlim(-0.8, len(common_vals) - 0.2)
    ax.set_ylim(bottom=0)
    ax.grid(True, axis="y", alpha=0.3, linestyle="--")
    ax.tick_params(axis="y", labelsize=11)
    ax.legend(fontsize=11, loc="upper right", framealpha=0.9, edgecolor="gray")

plt.savefig(OUTPUT_FIG_TXGAIN, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_FIG_TXGAIN}")
plt.show()


Saved → transfer_bar_txgain.png


In [37]:
# ─────────────────────────────────────────────────────────────
# Figure 6 — Power (W) vs Feature Value  (scatter, single row)
# ─────────────────────────────────────────────────────────────
OUTPUT_FIG_POWER = "transfer_power_scatter.png"

EXP_LIST    = list(EXPERIMENTS.keys())
COL_XLABELS = ["Gain (dB)", "MCS Index", "Airtime Ratio"]
COL_TITLES  = ["Transmission gain", "MCS", "Airtime"]

Y_MIN = min(df_orig["pm_power"].min(), df_clean["pm_power"].min()) - 0.5
Y_MAX = max(df_orig["pm_power"].max(), df_clean["pm_power"].max()) + 0.5

fig6, axes6 = plt.subplots(1, 3, figsize=(16, 5), constrained_layout=True)
fig6.suptitle(
    "Transfer Learning — Predicted vs Actual Power (W) by Feature Value\n"
    "Model3 (DNN + XGBoost)",
    fontsize=13, fontweight="bold"
)

for col_idx, exp_name in enumerate(EXP_LIST):
    ax = axes6[col_idx]

    p_s1 = power_scratch_D1[exp_name].sort_values("slice_val")
    p_s2 = power_scratch_D2[exp_name].sort_values("slice_val")
    p_cA = power_cross_A[exp_name].sort_values("slice_val")
    p_cB = power_cross_B[exp_name].sort_values("slice_val")

    # Each series plotted with its own available slice_vals — no intersection needed
    ax.scatter(p_s1["slice_val"], p_s1["mean_pred"],
               color="#1f77b4", marker="o", s=25, zorder=3, label="DS1 (no transfer)")
    ax.scatter(p_s2["slice_val"], p_s2["mean_pred"],
               color="#ff7f0e", marker="o", s=25, zorder=3, label="DS2 (no transfer)")
    ax.scatter(p_cA["slice_val"], p_cA["mean_pred"],
               color="#2ca02c", marker="^", s=30, zorder=3, label="DS1→DS2 (transfer)")
    ax.scatter(p_cB["slice_val"], p_cB["mean_pred"],
               color="#d62728", marker="^", s=30, zorder=3, label="DS2→DS1 (transfer)")
    # DS1 actual
    ax.scatter(p_s1["slice_val"], p_s1["mean_actual"],
               color="black", marker="x", s=40, linewidths=1.8, zorder=4,
               label="Experimental data")
    # DS2 actual — same style, no duplicate legend entry
    ax.scatter(p_s2["slice_val"], p_s2["mean_actual"],
               color="black", marker="x", s=40, linewidths=1.8, zorder=4)

    ax.set_title(COL_TITLES[col_idx], fontsize=11, fontweight="bold", pad=6)
    ax.set_xlabel(COL_XLABELS[col_idx], fontsize=10)
    ax.set_ylabel("Power (W)", fontsize=10)
    ax.set_ylim(Y_MIN, Y_MAX)
    ax.grid(True, alpha=0.3, linestyle="--")
    ax.tick_params(labelsize=9)
    ax.legend(fontsize=8.5, loc="upper left", framealpha=0.9, edgecolor="gray")

plt.savefig(OUTPUT_FIG_POWER, dpi=180, bbox_inches="tight")
print(f"Saved → {OUTPUT_FIG_POWER}")
plt.show()

Saved → transfer_power_scatter.png


In [40]:
# ─────────────────────────────────────────────────────────────
# Figure 8 -- CDF of Relative Error
#
# Left:  Target = Dataset 1
#   blue:   Source DS1 (no transfer, scratch_D1)
#   orange: Source DS2 -> DS1 (cross_B)
# Right: Target = Dataset 2
#   blue:   Source DS1 -> DS2 (cross_A)
#   orange: Source DS2 (no transfer, scratch_D2)
#
# Each line pools all experiments (Txgain + MCS + Airtime)
# ─────────────────────────────────────────────────────────────
OUTPUT_FIG_CDF = "transfer_cdf_re.png"

def collect_re(key):
    arrays = []
    for seed in SEEDS:
        for exp_name in EXPERIMENTS:
            arrays.append(all_re[seed][key][exp_name])
    return np.concatenate(arrays)

def plot_cdf(ax, re_arr, label, color, ls="-"):
    sorted_re = np.sort(re_arr)
    cdf = np.arange(1, len(sorted_re) + 1) / len(sorted_re)
    ax.plot(sorted_re, cdf, color=color, lw=2.2, linestyle=ls, label=label)

re_scratch_D1 = collect_re("scratch_D1")
re_cross_B    = collect_re("cross_B")
re_cross_A    = collect_re("cross_A")
re_scratch_D2 = collect_re("scratch_D2")

# Compute shared x-axis range across all series
x_max = max(re_scratch_D1.max(), re_cross_B.max(),
            re_cross_A.max(),    re_scratch_D2.max())

fig8, axes8 = plt.subplots(1, 2, figsize=(13, 5), constrained_layout=True,
                            sharex=True, sharey=True)
fig8.suptitle(
    "Transfer Learning -- CDF of Relative Error (%)\n"
    "Model3 (DNN + XGBoost)",
    fontsize=13, fontweight="bold"
)

# Left: Target DS1
plot_cdf(axes8[0], re_scratch_D1, "Source DS1 (no transfer)", "#1f77b4", ls="-")
plot_cdf(axes8[0], re_cross_B,    "Source DS2 -> DS1",        "#ff7f0e", ls="--")
axes8[0].set_title("Target = Dataset 1", fontsize=12, fontweight="bold")
axes8[0].set_xlabel("Relative Error (%)", fontsize=11)
axes8[0].set_ylabel("CDF", fontsize=11)
axes8[0].grid(True, alpha=0.3, linestyle="--")
axes8[0].legend(fontsize=10, framealpha=0.9, edgecolor="gray")

# Right: Target DS2
plot_cdf(axes8[1], re_cross_A,    "Source DS1 -> DS2",         "#1f77b4", ls="-")
plot_cdf(axes8[1], re_scratch_D2, "Source DS2 (no transfer)",  "#ff7f0e", ls="--")
axes8[1].set_title("Target = Dataset 2", fontsize=12, fontweight="bold")
axes8[1].set_xlabel("Relative Error (%)", fontsize=11)
axes8[1].grid(True, alpha=0.3, linestyle="--")
axes8[1].legend(fontsize=10, framealpha=0.9, edgecolor="gray")

# Apply shared axis limits
axes8[0].set_xlim(0, x_max)
axes8[0].set_ylim(0, 1.02)

plt.savefig(OUTPUT_FIG_CDF, dpi=180, bbox_inches="tight")
print(f"Saved -> {OUTPUT_FIG_CDF}")
plt.show()

Saved -> transfer_cdf_re.png


In [28]:
# ─────────────────────────────────────────────────────────────
# Table — Percentile Summary of MRE (%)
#
# Rows:    Target DS1 / Source DS1 (no transfer)
#          Target DS1 / Source DS2 -> DS1
#          Target DS2 / Source DS1 -> DS2
#          Target DS2 / Source DS2 (no transfer)
# Columns: 25th, Median, 75th, 90th, Max  (all in MRE %)
# ─────────────────────────────────────────────────────────────

PCTS = [25, 50, 75, 90, 100]
COL_NAMES = ["25th Percentile", "Median", "75th Percentile", "90th Percentile", "Max Value"]

rows = [
    ("Target Dataset 1", "Source DS1 (no transfer)", collect_re("scratch_D1")),
    ("Target Dataset 1", "Source DS2 -> DS1",         collect_re("cross_B")),
    ("Target Dataset 2", "Source DS1 -> DS2",          collect_re("cross_A")),
    ("Target Dataset 2", "Source DS2 (no transfer)",  collect_re("scratch_D2")),
]

index = pd.MultiIndex.from_tuples(
    [(t, s) for t, s, _ in rows],
    names=["Target", "Source"]
)

data = {
    col: [float(np.percentile(re, p)) for _, _, re in rows]
    for col, p in zip(COL_NAMES, PCTS)
}

df_table = pd.DataFrame(data, index=index).round(3)

print("Performance Metrics — MRE (%)")
print("=" * 80)
print(df_table.to_string())
print("=" * 80)

from IPython.display import display
display(df_table.style
    .set_caption("Comparison of Prediction Models based on CDFs of MRE (%)")
    .format("{:.3f}")
    .set_table_styles([
        {"selector": "caption", "props": [("font-weight", "bold"), ("font-size", "13px")]},
        {"selector": "th",      "props": [("text-align", "center")]},
        {"selector": "td",      "props": [("text-align", "center")]},
    ])
)


Performance Metrics — MRE (%)
                                           25th Percentile  Median  75th Percentile  90th Percentile  Max Value
Target           Source                                                                                        
Target Dataset 1 Source DS1 (no transfer)            0.757   1.520            2.478            3.616     13.121
                 Source DS2 -> DS1                   0.799   1.624            2.687            3.880     13.009
Target Dataset 2 Source DS1 -> DS2                   0.718   1.597            2.896            4.713     20.081
                 Source DS2 (no transfer)            0.690   1.445            2.632            3.982     17.203


In [29]:
# ─────────────────────────────────────────────────────────────
# Summary Table  (mean across seeds)
# ─────────────────────────────────────────────────────────────
print("\n" + "="*95)
print(f"SUMMARY: Cross-transfer vs No-transfer Scratch  (full data | seeds={SEEDS})")
print("="*95)
print(f"{'Experiment':<22} {'Direction':<38} {'MRE mean':>9} {'Max RE':>8} {'Min RE':>8} {'Δ MRE':>9}")
print("-"*95)

rows = [
    ("Target=DS1", "Cross-transfer DS-2→DS-1", cross_B,    scratch_D1),
    ("Target=DS1", "No-transfer DS-1 scratch",  scratch_D1, None),
    ("Target=DS2", "Cross-transfer DS-1→DS-2", cross_A,    scratch_D2),
    ("Target=DS2", "No-transfer DS-2 scratch",  scratch_D2, None),
]

for exp_name in EXPERIMENTS:
    print(f"\n{exp_name}")
    for target_lbl, dir_lbl, data_dict, baseline_dict in rows:
        st   = data_dict[exp_name]
        mre  = st["mean_re_mean"].mean()
        mxr  = st["max_re_mean"].max()
        mnr  = st["min_re_mean"].min()
        if baseline_dict is not None:
            base_mre = baseline_dict[exp_name]["mean_re_mean"].mean()
            delta    = base_mre - mre
            delta_str = f"{delta:+.3f}%"
        else:
            delta_str = "(baseline)"
        print(f"  [{target_lbl}] {dir_lbl:<36} {mre:>9.3f} {mxr:>8.2f} {mnr:>8.4f} {delta_str:>9}")

print("\n" + "="*95)
print("(+Δ = cross-transfer better than scratch)")


SUMMARY: Cross-transfer vs No-transfer Scratch  (full data | seeds=[42])
Experiment             Direction                               MRE mean   Max RE   Min RE     Δ MRE
-----------------------------------------------------------------------------------------------

Transmission Gain
  [Target=DS1] Cross-transfer DS-2→DS-1                 2.012    12.70   0.0002   -0.205%
  [Target=DS1] No-transfer DS-1 scratch                 1.806    13.11   0.0003 (baseline)
  [Target=DS2] Cross-transfer DS-1→DS-2                 2.159    19.19   0.0027   -0.252%
  [Target=DS2] No-transfer DS-2 scratch                 1.907    17.09   0.0062 (baseline)

MCS
  [Target=DS1] Cross-transfer DS-2→DS-1                 1.869    12.43   0.0030   -0.061%
  [Target=DS1] No-transfer DS-1 scratch                 1.808    12.53   0.0004 (baseline)
  [Target=DS2] Cross-transfer DS-1→DS-2                 2.296    16.59   0.0030   -0.427%
  [Target=DS2] No-transfer DS-2 scratch                 1.869    16.81   